In [1]:
import torch
import pandas as pd
from tqdm import tqdm
import traceback
import json
from src.extractors.extractor import (
    Extractor,
    BielikExtractor,
    DummyExtractor,
    PllumExtractor,
)
from src.extractors.openai import OpenAIExtractor
import os
from tenacity import retry, stop_after_attempt, wait_fixed

torch.cuda.empty_cache()

In [ ]:
DEBUG = True
model_name = "speakleash/Bielik-11B-v2.2-Instruct"

In [ ]:
acceptable_models = [
    "speakleash/Bielik-11B-v2.2-Instruct",
    "dummy",
    "CYFRAGOVPL/Llama-PLLuM-8B-instruct",
    "o3-mini",
]
assert (
    model_name in acceptable_models
), f"Model {model_name} not in acceptable models: {acceptable_models}"

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

extractors_map = {
    "speakleash/Bielik-11B-v2.2-Instruct": BielikExtractor,
    "dummy": DummyExtractor,
    "CYFRAGOVPL/Llama-PLLuM-8B-instruct": PllumExtractor,
}

if model_name == "o3-mini":
    extractor = OpenAIExtractor(
        model_name=model_name,
        batch_input_file_location="data/edc_baseline/openai_batch_file.jsonl",
    )
else:
    extractor: Extractor = extractors_map[model_name](device=device)
    model_size = extractor.get_memory_footprint()
    print(f"Model size: {model_size:.2f} GB")

In [5]:
relations_schema = pd.read_csv("data/edc_baseline/schema.csv", header=None)

relations_description = ""
for i in range(0, len(relations_schema)):
    relation_name, relation_description = relations_schema.iloc[i]
    relations_description += f"{relation_name} - {relation_description}\n"

In [6]:
with open("data/edc_baseline/oie_few_shot_examples.txt", "r") as file:
    fewshot_examples = "".join(file.readlines())

In [7]:
with open("data/edc_baseline/dataset.txt", "r") as file:
    dataset = file.readlines()

In [8]:
def create_system_content(relations_description, fewshot_examples):
    return f"""Twoim zadaniem jest wyciągnąć jedną relację łączącą dwa obiekty występujące w tekście, jako trójkę. Trójka musi być w postaci [[Poprzednik, Relacja, Następnik]]. Poprzednik i Następnik są wyrażeniami zapisanymi w tekście. Relacja jest krótkim zapisem związku, jaki łączy Poprzednik i Następnik.
W swojej odpowiedzi przedstaw dokładnie jedną trójkę. Jeśli w tekście jest więcej możliwych trójek, wybierz najardziej prawdopodobną. Nie podawaj żadnych innych informacji czy wyjaśnień.
            
Jedyne relacje, jakie możesz wyciągnąć, to:
{relations_description}

Poniżej przykłady zdań, w których występują obiekty, dla których należy wyciągnąć trójkę:
{fewshot_examples}"""

In [9]:
def create_prompt_content(sample):
    return f"""Tekst: {sample}
Trójka:"""

In [ ]:
responses = []
errors = []

system_content = create_system_content(relations_description, fewshot_examples)

if model_name == "o3-mini":
    for sample in tqdm(dataset[:10] if DEBUG else dataset):
        prompt_content = create_prompt_content(sample)
        extractor.add_request_to_batch(
            system_content=system_content, prompt_content=prompt_content
        )

    batch_id = extractor.run_batch("test").id
    print("Batch ID:", batch_id)

    os.remove(extractor.batch_input_file_location)

    bar = tqdm(total=10 if DEBUG else len(dataset))

    @retry(wait=wait_fixed(10), stop=stop_after_attempt(6 * 60 * 24))
    def ask_for_batch_status():
        batch = extractor.get_batch(batch_id=batch_id)

        if batch.status == "completed":
            print("Batch completed")

            result_file = extractor.client.files.content(batch.output_file_id)
            error_file = (
                extractor.client.files.content(batch.error_file_id)
                if batch.error_file_id
                else None
            )

            responses_list = []
            errors_list = []
            for line in result_file.iter_lines():
                responses_list.append(json.loads(line.strip()))

            if error_file:
                for line in error_file.iter_lines():
                    errors_list.append(json.loads(line.strip()))

            for data in extractor.requests:
                req_id = data["custom_id"]
                response = next(
                    (x for x in responses_list if x["custom_id"] == req_id), None
                )

                error = next((x for x in errors_list if x["custom_id"] == req_id), None)

                if response:
                    responses.append(
                        response["response"]["body"]["choices"][0]["message"]["content"]
                    )
                elif error:
                    errors.append(error)
        else:
            bar.n = batch.request_counts.completed + batch.request_counts.failed
            bar.update(0)
            raise Exception("Batch is still in progress")

    ask_for_batch_status()

else:
    for sample in tqdm(dataset[:10] if DEBUG else dataset):
        try:
            print("Sample:", sample.strip())

            template = extractor.create_messages_template(
                system_content, create_prompt_content(sample)
            )

            response = extractor.get_response_text(template)
            responses.append(response)
            print("Response:", response)
        except Exception as e:
            tb = traceback.format_exc()
            error_message = f"Error for sample: {sample.strip()}\n{tb}\n"
            errors.append(error_message)
            responses.append("Error\n")
            print("Error:", error_message)

In [ ]:
with open("data/edc_baseline/extracted_relations.json", "w", encoding="utf-8") as file:
    json.dump(dict(responses=responses), file, ensure_ascii=False, indent=2)

with open("data/edc_baseline/errors.json", "w", encoding="utf-8") as file:
    json.dump(dict(errors=errors), file, ensure_ascii=False, indent=2)